# C05 单因子到多因子

本节目标：完成因子检验与多因子整合的最小流程。


## 研究流程
- 清洗：winsorize + z-score
- 单因子：RankIC + 分组收益
- 多因子：滚动线性回归


In [ ]:
MODE = "offline"
START_DATE = "2019-01-31"
END_DATE = "2022-12-31"
UNIVERSE_SIZE = 80
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(SEED)


In [ ]:
# 构造月频截面样本用于教学演示
dates = pd.date_range(START_DATE, END_DATE, freq="ME")
assets = [f"S{i:03d}" for i in range(1, UNIVERSE_SIZE + 1)]
rows = []

for d in dates:
    for a in assets:
        value = np.random.normal()
        quality = np.random.normal()
        momentum = np.random.normal()
        noise = np.random.normal(scale=0.05)
        # 让 future_ret 与三个因子有可识别关系
        future_ret = 0.03 * value + 0.02 * quality + 0.04 * momentum + noise
        rows.append({
            "date": d,
            "asset": a,
            "value": value,
            "quality": quality,
            "momentum": momentum,
            "future_ret": future_ret,
        })

panel = pd.DataFrame(rows)
panel.head()


## 单因子清洗与检验
课堂重点：
- 逐日期去极值
- 逐日期标准化
- 用 RankIC 与 Q5-Q1 双重验证


In [ ]:
def winsorize(s, n=3):
    m, sd = s.mean(), s.std()
    return s.clip(m - n * sd, m + n * sd)


def zscore(s):
    return (s - s.mean()) / (s.std() + 1e-12)

pc = panel.copy()
for col in ["value", "quality", "momentum"]:
    pc[col] = pc.groupby("date")[col].transform(winsorize)
    pc[col] = pc.groupby("date")[col].transform(zscore)

rank_ic = pc.groupby("date")[["momentum", "future_ret"]].apply(
    lambda x: x["momentum"].rank().corr(x["future_ret"].rank())
)

pc["q"] = pc.groupby("date")["momentum"].transform(lambda x: pd.qcut(x.rank(method="first"), 5, labels=False))
qret = pc.groupby(["date", "q"])["future_ret"].mean().unstack()
ls = qret[4] - qret[0]
ls_nav = (1 + ls).cumprod()

rank_ic.describe().round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ls_nav.plot(ax=ax, title="C05 单因子多空净值")
ax.set_ylabel("NAV")
plt.show()


## 多因子滚动回归
- 每月只用过去 12 个月样本训练
- 用预测值分组构建组合


In [ ]:
factors = ["value", "quality", "momentum"]
all_dates = sorted(pc["date"].unique())
preds = []

for i in range(12, len(all_dates)):
    train_dates = all_dates[i - 12:i]
    test_date = all_dates[i]
    tr = pc[pc["date"].isin(train_dates)]
    te = pc[pc["date"] == test_date].copy()

    X = tr[factors].values
    y = tr["future_ret"].values
    beta = np.linalg.lstsq(np.c_[np.ones(len(X)), X], y, rcond=None)[0]

    te["pred_ret"] = np.c_[np.ones(len(te)), te[factors].values] @ beta
    preds.append(te[["date", "asset", "pred_ret", "future_ret"]])

pred = pd.concat(preds, ignore_index=True)
pred["pred_q"] = pred.groupby("date")["pred_ret"].transform(lambda x: pd.qcut(x.rank(method="first"), 5, labels=False))

mf = pred.groupby(["date", "pred_q"])["future_ret"].mean().unstack()[4]
mf_nav = (1 + mf).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
(100 * mf_nav / mf_nav.iloc[0]).plot(ax=ax, title="C05 多因子策略净值")
ax.set_ylabel("Index=100")
plt.show()

report = pd.DataFrame({
    "rank_ic_mean": [rank_ic.mean()],
    "rank_ic_ir": [rank_ic.mean() / (rank_ic.std() + 1e-12)],
    "single_factor_ann_ret": [ls.mean() * 12],
    "multi_factor_ann_ret": [mf.mean() * 12],
}).round(4)
report


## 小结与练习
- 单因子负责“发现信息”，多因子负责“融合信息”。
- 练习：移除 momentum 因子后复跑，比较指标变化。
